# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [3]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [4]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [5]:
import torch

# 1. 创建三个浮点标量参数 w1, w2, b，并启用梯度
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# 2. 一起交给优化器（SGD），设置初始学习率
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

# 开始 2000 轮训练循环
for epoch in range(1, 2001):
    # 动态计算本轮学习率：0.1 / (epoch ** 0.5)
    current_lr = 0.1 / (epoch ** 0.5)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    # 清空旧梯度
    optimizer.zero_grad()

    # 调用 predict 函数计算 y_pred（传入 x1, x2 和当前参数）
    y_pred = predict(x1, x2, w1, w2, b)

    # 计算逐样本绝对误差（L1 Loss）
    loss_each = torch.abs(y_pred - y)
    
    # 计算平均损失（得到一个标量）
    loss_mean = loss_each.mean()

    # 反向传播，计算梯度
    loss_mean.backward()

    # 更新参数
    optimizer.step()

# 3. 训练结束后，在 torch.no_grad() 环境下重新计算并打印
with torch.no_grad():
    # 用训练好的参数重新预测
    y_pred_final = predict(x1, x2, w1, w2, b)
    
    # 重新计算逐样本绝对误差和平均损失
    loss_each_final = torch.abs(y_pred_final - y)
    loss_mean_final = loss_each_final.mean()

    # 按照要求打印所有内容及其 shape
    print(f"训练后的参数: w1={w1.item():.4f}, w2={w2.item():.4f}, b={b.item():.4f}")
    print(f"x1: {x1}, shape: {tuple(x1.shape)}")
    print(f"x2: {x2}, shape: {tuple(x2.shape)}")
    print(f"y (目标): {y}, shape: {tuple(y.shape)}")
    print(f"y_pred (预测): {y_pred_final}, shape: {tuple(y_pred_final.shape)}")
    print(f"逐样本损失: {loss_each_final}, shape: {tuple(loss_each_final.shape)}")
    print(f"平均损失: {loss_mean_final.item():.4f}, shape: {tuple(loss_mean_final.shape)}")

训练后的参数: w1=1.9990, w2=-3.0010, b=0.9994
x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]), shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]), shape: (6,)
y (目标): tensor([-6., -1., -5.,  6., -4.,  4.]), shape: (6,)
y_pred (预测): tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954]), shape: (6,)
逐样本损失: tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]), shape: (6,)
平均损失: 0.0024, shape: ()


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

待填写。
